In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, count, sum, col
from pyspark.sql.types import StringType

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,None,pyspark,idle,,,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

#### In CSV when same location contains multiple files with different schema, it will show all the columns with null without merge schema

In [2]:
path = "s3://shivchoudhury-datasets/MergeSchema/customers/"

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
customer_df = spark.read.option("inferSchema", "true").option("header", "true").csv(path)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
customer_df.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------+----------+----------+---------+---------+-----------+----------+-------------+--------------------+----------------+
|CUSTOMER_ID|SALUTATION|FIRST_NAME|LAST_NAME|BIRTH_DAY|BIRTH_MONTH|BIRTH_YEAR|BIRTH_COUNTRY|       EMAIL_ADDRESS|CUSTOMER_RATINGS|
+-----------+----------+----------+---------+---------+-----------+----------+-------------+--------------------+----------------+
|   IZUVVII5|       Mr.|      John|    Davis|       24|          3|      2001|          USA|john.davis@exampl...|               3|
|   58NMDI8J|      Mrs.|     Casey|Rodriguez|        9|          2|      1981|      Germany|casey.rodriguez@t...|               2|
|   Y5IY9NKE|       Dr.|      Jane|    Jones|       28|         11|      1967|        China|jane.jones@exampl...|               5|
|   OMGWY5SZ|      Mrs.|      Jane|   Garcia|       18|          8|      2004|           UK|jane.garcia@mail.com|               2|
|   ZX822CPA|     Prof.|     Riley| Williams|       23|          4|      1961|     

In [ ]:
customer_df.write.option("mode", "overwrite").option("maxRecordsPerFile", 10).option("header", "true").csv("s3://shivchoudhury-datasets/MergeSchema/customers1/")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [44]:
def mask_mail(email):
    lst = email.split("@")
    first_char = lst[0][0]
    email = first_char + '*' * len(lst[0]) + '@' + lst[1]
    return email

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [45]:
masking = udf(mask_mail, StringType())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [46]:
customer_df = customer_df.withColumn('EMAIL_ADDRESS', masking(customer_df.EMAIL_ADDRESS))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [76]:
customer_df.select('SALUTATION').distinct().show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+
|SALUTATION|
+----------+
|      Mrs.|
|       Mr.|
|     Prof.|
|       Ms.|
|       Dr.|
+----------+

In [57]:
customer_df.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------+----------+----------+---------+---------+-----------+----------+-------------+--------------------+----------------+
|CUSTOMER_ID|SALUTATION|FIRST_NAME|LAST_NAME|BIRTH_DAY|BIRTH_MONTH|BIRTH_YEAR|BIRTH_COUNTRY|       EMAIL_ADDRESS|CUSTOMER_RATINGS|
+-----------+----------+----------+---------+---------+-----------+----------+-------------+--------------------+----------------+
|   IZUVVII5|       Mr.|      John|    Davis|       24|          3|      2001|          USA|john.davis@exampl...|               3|
|   58NMDI8J|      Mrs.|     Casey|Rodriguez|        9|          2|      1981|      Germany|casey.rodriguez@t...|               2|
|   Y5IY9NKE|       Dr.|      Jane|    Jones|       28|         11|      1967|        China|jane.jones@exampl...|               5|
|   OMGWY5SZ|      Mrs.|      Jane|   Garcia|       18|          8|      2004|           UK|jane.garcia@mail.com|               2|
|   ZX822CPA|     Prof.|     Riley| Williams|       23|          4|      1961|     

In [58]:
stats_df = customer_df.groupBy('BIRTH_COUNTRY').pivot("SALUTATION").count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [59]:
stats_df.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------+---+---+----+---+-----+
|BIRTH_COUNTRY|Dr.|Mr.|Mrs.|Ms.|Prof.|
+-------------+---+---+----+---+-----+
|        China| 48| 49|  39| 28|   32|
|      Germany| 32| 42|  46| 45|   54|
|        India| 42| 39|  32| 30|   41|
|       Brazil| 43| 35|  36| 41|   48|
|       Canada| 38| 38|  34| 38|   49|
|        Japan| 33| 30|  39| 53|   42|
|           UK| 47| 35|  44| 41|   34|
|       France| 49| 31|  46| 32|   42|
|          USA| 44| 48|  38| 35|   36|
|    Australia| 50| 36|  35| 40|   41|
+-------------+---+---+----+---+-----+

In [60]:
stats_df = customer_df.groupBy('BIRTH_COUNTRY').pivot("SALUTATION").agg(count("*").alias("total_count"))

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [61]:
stats_df.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------------+---+---+----+---+-----+
|BIRTH_COUNTRY|Dr.|Mr.|Mrs.|Ms.|Prof.|
+-------------+---+---+----+---+-----+
|        China| 48| 49|  39| 28|   32|
|      Germany| 32| 42|  46| 45|   54|
|        India| 42| 39|  32| 30|   41|
|       Brazil| 43| 35|  36| 41|   48|
|       Canada| 38| 38|  34| 38|   49|
|        Japan| 33| 30|  39| 53|   42|
|           UK| 47| 35|  44| 41|   34|
|       France| 49| 31|  46| 32|   42|
|          USA| 44| 48|  38| 35|   36|
|    Australia| 50| 36|  35| 40|   41|
+-------------+---+---+----+---+-----+

### Parquet

#### In Parquet when different files has different schemas in a location it will only show common location while trying to read, when we use mergeSchema it will show others columns also with null values

In [51]:
def csvToParquet(df_source_path, df_target_path):
    df = spark.read.option("inferSchema", "true").option("header", "true").csv(df_source_path)
    df.write.mode("append").parquet(df_target_path)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [62]:
# csvToParquet('s3://shivchoudhury-datasets/MergeSchema/customers/landing_customer_data_1.csv', 's3://shivchoudhury-datasets/MergeSchema/paquet-customers/')
# csvToParquet('s3://shivchoudhury-datasets/MergeSchema/customers/landing_customer_data_2.csv', 's3://shivchoudhury-datasets/MergeSchema/paquet-customers/')
# csvToParquet('s3://shivchoudhury-datasets/MergeSchema/customers/landing_customer_data_3.csv', 's3://shivchoudhury-datasets/MergeSchema/paquet-customers/')
# csvToParquet('s3://shivchoudhury-datasets/MergeSchema/customers/landing_customer_data_4.csv', 's3://shivchoudhury-datasets/MergeSchema/paquet-customers/')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [64]:
df = spark.read.option('inferSchema', 'true').parquet('s3://shivchoudhury-datasets/MergeSchema/paquet-customers/')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [65]:
df.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------+----------+----------+---------+---------+-----------+----------+-------------+--------------------+
|CUSTOMER_ID|SALUTATION|FIRST_NAME|LAST_NAME|BIRTH_DAY|BIRTH_MONTH|BIRTH_YEAR|BIRTH_COUNTRY|       EMAIL_ADDRESS|
+-----------+----------+----------+---------+---------+-----------+----------+-------------+--------------------+
|   4YZCNG66|     Prof.|      Alex|    Brown|       15|          6|      1960|    Australia| alex.brown@test.org|
|   4EYG05GW|     Prof.|      Alex| Martinez|        7|          4|      2003|       Brazil|alex.martinez@com...|
|   PI7I3BQ1|      Mrs.|      John|   Miller|       24|          3|      1992|      Germany|john.miller@compa...|
|   YX2Y51U5|       Ms.|    Morgan|    Davis|       25|          6|      1966|        China|morgan.davis@comp...|
|   IHAQFM5I|     Prof.|     Chris|Rodriguez|        6|         10|      1983|    Australia|chris.rodriguez@t...|
|   0N18MDGN|     Prof.|      Alex| Martinez|       14|          7|      1982|        In

In [66]:
df.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

2000

In [67]:
df = spark.read.option('inferSchema', 'true').option('mergeSchema', 'true').parquet('s3://shivchoudhury-datasets/MergeSchema/paquet-customers/')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [68]:
df.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------+----------+----------+---------+---------+-----------+----------+-------------+--------------------+----------------+
|CUSTOMER_ID|SALUTATION|FIRST_NAME|LAST_NAME|BIRTH_DAY|BIRTH_MONTH|BIRTH_YEAR|BIRTH_COUNTRY|       EMAIL_ADDRESS|CUSTOMER_RATINGS|
+-----------+----------+----------+---------+---------+-----------+----------+-------------+--------------------+----------------+
|   4YZCNG66|     Prof.|      Alex|    Brown|       15|          6|      1960|    Australia| alex.brown@test.org|               2|
|   4EYG05GW|     Prof.|      Alex| Martinez|        7|          4|      2003|       Brazil|alex.martinez@com...|               2|
|   PI7I3BQ1|      Mrs.|      John|   Miller|       24|          3|      1992|      Germany|john.miller@compa...|               3|
|   YX2Y51U5|       Ms.|    Morgan|    Davis|       25|          6|      1966|        China|morgan.davis@comp...|               2|
|   IHAQFM5I|     Prof.|     Chris|Rodriguez|        6|         10|      1983|    A

In [71]:
df.filter(col('CUSTOMER_RATINGS').isNotNull()).count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

1000

In [72]:
df.filter(col('CUSTOMER_RATINGS').isNull()).count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

1000

In [73]:
df.where(col('CUSTOMER_RATINGS').isNull()).show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----------+----------+----------+---------+---------+-----------+----------+-------------+--------------------+----------------+
|CUSTOMER_ID|SALUTATION|FIRST_NAME|LAST_NAME|BIRTH_DAY|BIRTH_MONTH|BIRTH_YEAR|BIRTH_COUNTRY|       EMAIL_ADDRESS|CUSTOMER_RATINGS|
+-----------+----------+----------+---------+---------+-----------+----------+-------------+--------------------+----------------+
|   3P4XQRIU|       Mr.|     Riley|  Johnson|       10|          2|      1967|          USA|riley.johnson@mai...|            null|
|   IA7V18FL|       Dr.|     Riley| Martinez|       11|          3|      1979|    Australia|riley.martinez@ma...|            null|
|   F9GLDZ3R|       Ms.|      Alex|    Brown|       22|          1|      2004|        India| alex.brown@mail.com|            null|
|   8I6UWXH0|     Prof.|      Alex| Martinez|        9|         12|      2001|        China|alex.martinez@tes...|            null|
|   K53VIUBM|       Ms.|       Sam|    Jones|       16|          1|      1963|     